# Airline Delay Classification with MLflow

Refer to this [github project](https://github.com/JaHerbas/Predicting_Flight_Delays).

In [1]:
# --- Import modules ---

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

import mlflow
import mlflow.sklearn
import os

/opt/Miniforge3-25.9.1-0-Linux-x86_64/envs/ptl-cuda-12-1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Set logging level ---

import warnings
warnings.filterwarnings("ignore")

## Preprocessing

In [3]:
# --- Load dataset ---
DATA_DIR = os.environ.get("CIML26_DATA_DIR") 
filename = DATA_DIR + "/Airline_2016_only.csv"
df = pd.read_csv(filename)

In [4]:
# --- Create target ---

df['DELAYED'] = (df['ARR_DELAY'] > 0).astype(int)

In [5]:
# --- Choose features ---

columns_to_remove = [
    'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN',
    'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED',
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'CARRIER_DELAY', 'WEATHER_DELAY',
    'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'Unnamed: 27'
]
df.drop(columns=columns_to_remove, inplace=True)

In [6]:
# --- Preprocessing ---

categorical_features = ['OP_CARRIER', 'ORIGIN', 'DEST']
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.difference(['DELAYED'])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [7]:
# --- Split dataset ---

X = df.drop(columns=['DELAYED', 'FL_DATE'])
y = df['DELAYED']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Define Models

In [8]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=20, max_depth=4, random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss', n_estimators=20, max_depth=3, use_label_encoder=False, random_state=42),
}

## Using MLflow

1. **Run the launch_mlflow.sh script**:

    In the terminal, make sure you're in the same directory as the launch_mlflow.sh file then run the following
   
   ```bash
   bash launch_mlflow.sh
   ```

The launch_mlflow.sh script starts an MLflow server on a random available port and connects it to Expanse's reverse proxy service, giving you a unique public URL to access the MLflow UI in your browser every time. It should look like this
   
   ```bash
   https://<to-be-replaced>.expanse-user-content.sdsc.edu
   ```
   
The server will keep running until your job ends, at which point your run data is saved automatically.

   
2. **In the notebook**:

    Set the tracking uri to be the generated link
   ```python
   mlflow.set_tracking_uri('https://<to-be-replaced>.expanse-user-content.sdsc.edu')
   ```

   This allows MLflow to log runs to the generated link.

3. In the UI, navigate to the experiment. Make sure you're on Model Training and not GenAI in the sidebar. Then click on any run. You can see graphs with your experiment metrics under the `Model Metrics` tab


In [9]:
# --- Initialize MLflow ---

# TO-DO - 
mlflow.set_tracking_uri('https://stereo-palm-exhume.expanse-user-content.sdsc.edu')
mlflow.set_experiment("airline-delay-classification")

<Experiment: artifact_location='/home/mhnguyen/Teaching/CIML2026/mhn-ciml/mlops/mlruns/3', creation_time=1782154287441, experiment_id='3', last_update_time=1782154287441, lifecycle_stage='active', name='airline-delay-classification', tags={}, trace_location=None, workspace='default'>

## Train and Evaluate Models

In [10]:
results = {}

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)

        # Log model, parameters, and metrics
        mlflow.sklearn.log_model(pipeline, name="model")
        mlflow.log_params(model.get_params())
        mlflow.log_metric("accuracy", accuracy)
        
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, metric_value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", metric_value)

        results[model_name] = {'accuracy': accuracy}

2026/06/23 13:41:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3/runs/296febe7144a404880894b860c978b38
🧪 View experiment at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3


2026/06/23 13:42:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Decision Tree at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3/runs/c59aa629bbe54243a884753156368121
🧪 View experiment at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3


2026/06/23 13:42:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3/runs/cc38718b024f4c679be67d755763d8b8
🧪 View experiment at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3


2026/06/23 13:42:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3/runs/723f38f9b7db4fb992f1a5336ad23fe5
🧪 View experiment at: https://stereo-palm-exhume.expanse-user-content.sdsc.edu/#/experiments/3


In [11]:
# --- Display results ---

results_df = pd.DataFrame(results).T
results_df

,accuracy
Logistic Regression,0.664003
Decision Tree,0.664067
Random Forest,0.664004
XGBoost,0.672714
